# VoxCPM2 Studio | Kaggle

Activa **GPU T4 ×2** e **Internet** antes de ejecutar. Este notebook sirve para **Save Version → Save & Run All**. La última celda mantiene Gradio en primer plano; el enlace `gradio.live` es temporal y no es hosting permanente. El trabajo corre en la ejecución versionada mientras la sesión permanezca activa.

In [ ]:
from pathlib import Path
import shutil, subprocess, os
ROOT = Path("/kaggle/working/voxcpm2-dataset-studio")
REPO = "https://github.com/ThowiLabs/VoxCPM2-Studio.git"
if not (ROOT / "studio.py").exists():
    # Supports both a GitHub clone (after publication) and a ZIP attached to the Kaggle notebook.
    zipfiles = list(Path("/kaggle/input").rglob("VoxCPM2-Studio-GIT.zip"))
    if zipfiles:
        import zipfile
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zipfiles[0]) as z:
            z.extractall(ROOT)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
assert (ROOT / "studio.py").exists()
print("Proyecto:", ROOT, flush=True)

In [ ]:
import subprocess
subprocess.run(["bash", "install_kaggle.sh"], cwd=str(ROOT), check=True)
PYTHON = ROOT / ".venv/bin/python"
subprocess.run([str(PYTHON), "-c", "import torch; print('CUDA:', torch.cuda.is_available()); print('Dispositivos:',torch.cuda.device_count()); assert torch.cuda.is_available(), 'Activa la GPU en Kaggle'"], check=True)
subprocess.run([str(PYTHON), "-m", "pytest", "-q", "test_studio.py"], cwd=str(ROOT), check=True)

## Servidor Gradio, última celda

**No uses** `Popen` ni `nohup`. En `Save & Run All` se verá la URL pública en los logs de la ejecución. **Sin autenticación por defecto**; si cargas voces privadas, añade `STUDIO_USER` y `STUDIO_PASSWORD` como secretos de Kaggle.

In [ ]:
import os, subprocess
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["MPLBACKEND"] = "Agg"
# Remove credentials inherited from earlier notebooks, unless explicitly configured in Secrets.
env.pop("STUDIO_USER", None)
env.pop("STUDIO_PASSWORD", None)
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    values = {}
    for key in ("STUDIO_USER", "STUDIO_PASSWORD"):
        try:
            values[key] = client.get_secret(key)
        except Exception:
            pass
    if values.get("STUDIO_USER") and values.get("STUDIO_PASSWORD"):
        env.update(values)
except ImportError:
    pass
print("VoxCPM2 Studio: iniciando Gradio --share en primer plano...", flush=True)
subprocess.run([str(PYTHON), "-u", "studio.py", "--share", "--port", "7860"],
               cwd=str(ROOT), env=env, check=True)